 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submisison 
Name           : Selvi Ganesh Kumar                <br>
Student Number : L00196360          <br>
Due Date       : 12th May 2026                <br>
Assignment     :             <br>
Module         : AI for Vision and NLP    <br>
Course         : Postgraduate Diploma in Big Data Analytics and AI

## NLP and Vision Pipeline : High Level
An image of your working pipeline at high level can be inserted here



# Initialisation
Perform pip installs(or use a requirements.txt) <br>
perform imports

## Install packages

In [5]:
# pip installs

requirements = """
opencv-python
pytesseract
numpy
matplotlib
pandas
pymupdf
easyocr
tqdm
pillow
"""

with open("requirements.txt", "w") as f:
    f.write(requirements.strip())

In [6]:
# pip install -r requirements.txt

## Imports

In [8]:
# imports

import os
import re
import zipfile

import numpy as np
import pandas as pd

import cv2
import matplotlib.pyplot as plt

import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

import pymupdf 
import fitz

from tqdm import tqdm

# Support Functions

In [10]:
# code here

def preprocess_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    thresh = cv2.threshold(blur, 150, 255, cv2.THRESH_BINARY)[1]
    return thresh

# NLP

## Sub Heading 1

## code

In [13]:
# The dataset has different filtypes: .jpg (images), .bmp (images) and .pdf (documents)
# ZIP → extract → detect file type → extract text → NLP → output

### Image loading

In [15]:
# STEP 1: Extract ZIP
import zipfile
import os

path = r"C:\Education\M.Sc Documents\Computer Vision & NLP\CA2\documents_ca2.zip"

with zipfile.ZipFile(path, "r") as zip_ref:
    zip_ref.extractall("docs")

In [16]:
# To view the extracted files from the zip.

folder = "docs"

for root, dirs, files in os.walk(folder):
    print("Folder:", root)
    print("Files:", files)
  

Folder: docs
Files: []
Folder: docs\.ipynb_checkpoints
Files: []
Folder: docs\documents_ca2
Files: ['aloevera_back.jpg', 'cartoon_drawing.pdf', 'drawing_using_words.bmp', 'know_receipe.pdf', 'research_paper.pdf', 'scikit_learn.pdf', 'trinity_brochure.pdf']


In [17]:
# step 2: Separate file types and storing them in their own variety, for further processing

jpg_files = []
bmp_files = []
pdf_files = []

for root, dirs, files in os.walk(folder):
    for file in files:
        path = os.path.join(root, file)

        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            jpg_files.append(path)

        elif file.lower().endswith(".bmp"):
            bmp_files.append(path)

        elif file.lower().endswith(".pdf"):
            pdf_files.append(path)

print("JPG:", len(jpg_files))
print("BMP:", len(bmp_files))
print("PDF:", len(pdf_files))

JPG: 1
BMP: 1
PDF: 5


### OCR using Tesseract on images

In [19]:
# step 3: storing the images in the images files to process images (ocr)

image_files = jpg_files + bmp_files

results = []

for file in image_files:
    img = cv2.imread(file)

    processed = preprocess_image(img)
    custom_config = r'--oem 1 --psm 6'  

    text = pytesseract.image_to_string(processed, config = custom_config)

    results.append({
        "file": file,
        "type": "image",
        "text": text
    })



In this project, a hybrid approach was adopted for processing PDF documents to ensure robust and accurate text extraction across different document types. PDFs can broadly be categorised into two types: text-based PDFs, which contain embedded digital text, and scanned PDFs, which are essentially images of documents and do not contain extractable text. To handle both cases effectively, the system first attempts to extract text directly using PyMuPDF, which is efficient and preserves formatting when digital text is available. If no text is detected on a page (i.e., the extracted content is empty or minimal), the system assumes the page is image-based and applies Optical Character Recognition (OCR) using Tesseract. In this case, the PDF page is converted into an image and processed to extract text. This hybrid strategy improves both performance and accuracy by avoiding unnecessary OCR on text-based documents while still enabling text extraction from scanned or image-based PDFs. It also demonstrates a practical integration of document analysis techniques, aligning with real-world multi-modal document understanding systems.

In [21]:
# handling the pdf files separately, either by converting into image for scanned pdf, or by extracting the text directly, 
# incase of read text based pdf, them the text is extracted:


for file in pdf_files:
    doc = fitz.open(file)

    text = ""  # one full document text

    for page in doc:
        extracted = page.get_text()

        if extracted.strip():
            text += extracted + "\n"

        else:
            pix = page.get_pixmap(dpi=200)

            img = np.frombuffer(pix.samples, dtype=np.uint8)
            img = img.reshape(pix.height, pix.width, pix.n)

            custom_config = r'--oem 1 --psm 6'
            text += pytesseract.image_to_string(img, config=custom_config) + "\n"

    results.append({
        "file": file,
        "type": "pdf",
        "text": text
    })

### Basic text extraction from visual data

In [23]:
results


[{'file': 'docs\\documents_ca2\\aloevera_back.jpg',
  'type': 'image',
  'text': 'See WET JUICE, weit; criabies US to capture trie\nnatural properties of the Aloe vera plant. This\ncooling gel nourishes, soothes and can help |\nrestore dry and problem skin. It can also be\napplied to stretch marks, scars and dry or sun\nexposed skin. |\nNatural actives: Organic Aloe Vera.\nBeneficial properties: Soothing, Cooling,\nRestoring, Nourishing\nDirection ‘ | in, Repeat as often a:\nrequired Over the bod,\n(supery :\nIndicazi § 5 senie la necessit.\n(supervis, ad\nAvvertenz a, =. /contatts con gh accel\n“so ch COM au OLEH Hate, MIO GuUTe cul abbcidante acqua \'\nt POCare SU Cute lesa. IN Caso di rritazione cutanea sospendere «\nconte Tuttliz20 di questo prodotto se. si @ allergic’ a qu.’\non oonente, Tenere lontano dalla portata dei bambini. Conserv"\n4090 fresco e€ asciutto, al riparo dalla luce.\n}\n| Aloe Vera Inner Gel —\n\\ Not fo, intemal\n\\ Ne Water, Dg not Avoid contact with eyes. Ini

This pipeline processes a mixed dataset of images and PDF files to extract text for further NLP use. PDF files are handled using PyMuPDF, which directly extracts embedded digital text, resulting in highly accurate outputs. Image files are processed using Tesseract OCR, which converts visual content into machine-readable text. The quality of extracted text from images varies depending on the type of image: clear printed images (such as charts, diagrams, or scanned pages) produce better results, while handwritten images, blurred photos, or noisy scans often lead to incomplete or incorrect recognition due to OCR limitations. Therefore, differences in output quality are not caused by errors in the code, but by the nature and quality of the input data and the constraints of OCR technology when working with complex visual content.

Most of the images include scanned documents, photos (not scans), handwriting, artistic layouts, low contrast backgrounds. Without strong preprocessing, Tesseract struggles.

Next steps: Preprocessing, and improve config.

#### Creating a dataframe for the purpose of organising the above results

In [25]:
results_df = pd.DataFrame(results)

results_df

,file,type,text
0,docs\documents_ca2\aloevera_back.jpg,image,"See WET JUICE, weit; criabies US to capture tr..."
1,docs\documents_ca2\drawing_using_words.bmp,image,Yt She\nSe A ge ee 9 tok gi aa\nvie? i Been wh...
2,docs\documents_ca2\cartoon_drawing.pdf,pdf,4\nLesson\nFamous Artists Cartoon Course\nThe ...
3,docs\documents_ca2\know_receipe.pdf,pdf,Soups Recipe Book \n1 \n\nContents \nRecipes \...
4,docs\documents_ca2\research_paper.pdf,pdf,\nFig. 1. Classification algorithms \nThe ide...
5,docs\documents_ca2\scikit_learn.pdf,pdf,"05/05/2026, 22:21 1.1. Linear Models — scikit-..."
6,docs\documents_ca2\trinity_brochure.pdf,pdf,Undergraduate \nProspectus 2026\n\nContents \n...


The dataset contains a collection of mixed document types, including images and PDF files, all processed into a unified format for analysis. Each row represents a single document instance where the extracted text is obtained using OCR for images and a combination of direct text extraction and OCR fallback for PDFs. This allows both visual and textual content to be converted into machine-readable form. The resulting structure supports further NLP tasks such as text cleaning, feature extraction, and document understanding across heterogeneous sources within a single consistent dataset.

## Experiment 1 : Results:
represents the initial stage of the project, focusing on basic text extraction from images and PDF documents using OCR and document parsing techniques. This forms the foundation for later Vision preprocessing, NLP analysis, and multimodal integration. 

(Including the page numbers for all the pdf. so, each page in the pdf is a row.)

In [28]:
image_files = jpg_files + bmp_files

results = []

for file in image_files:
    img = cv2.imread(file)

    processed = preprocess_image(img)
    custom_config = r'--oem 1 --psm 6'  

    text = pytesseract.image_to_string(processed, config = custom_config)

    results.append({
        "file": file,
        "page": 0,
        "type": "image",
        "text": text
    })


#__________________________________

for file in pdf_files:
    doc = fitz.open(file)

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()

        # If text exists
        if text.strip():
            extracted_text = text

        # fallback OCR
        else:
            pix = page.get_pixmap(dpi=200)
            img = np.frombuffer(pix.samples, dtype=np.uint8)
            img = img.reshape(pix.height, pix.width, pix.n)

            custom_config = r'--oem 1 --psm 6'
            extracted_text = pytesseract.image_to_string(img, config=custom_config)

        results.append({
            "file": file,
            "page": page_num,
            "type": "pdf",
            "text": extracted_text
        })

In [29]:
results1_df = pd.DataFrame(results)

results1_df

,file,page,type,text
0,docs\documents_ca2\aloevera_back.jpg,0,image,"See WET JUICE, weit; criabies US to capture tr..."
1,docs\documents_ca2\drawing_using_words.bmp,0,image,Yt She\nSe A ge ee 9 tok gi aa\nvie? i Been wh...
2,docs\documents_ca2\cartoon_drawing.pdf,1,pdf,4\nLesson\nFamous Artists Cartoon Course\nThe ...
3,docs\documents_ca2\cartoon_drawing.pdf,2,pdf,Lesson\nFamous Artists Cartoon Course\nThe com...
4,docs\documents_ca2\cartoon_drawing.pdf,3,pdf,10\nAll cartoon characters are based on real p...
5,docs\documents_ca2\know_receipe.pdf,1,pdf,Soups Recipe Book \n1 \n
6,docs\documents_ca2\know_receipe.pdf,2,pdf,Contents \nRecipes \nQuick & Easy Soups – 20 m...
7,docs\documents_ca2\know_receipe.pdf,3,pdf,Jackfruit \nWhy not try going \nmeat free one ...
8,docs\documents_ca2\know_receipe.pdf,4,pdf,White potatoes \nFOR \nSweet potatoes \nKidney...
9,docs\documents_ca2\know_receipe.pdf,5,pdf,Creamy Pea Soup \nPerfect mid-week lunch when ...


The current system is in the early exploratory stage, focusing on basic OCR-based extraction from images and PDFs. This forms the foundation for future Vision preprocessing, NLP analysis, and multimodal integration.

In [31]:
# step 6: NLP

texts = [item["text"] for item in results]

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(texts)

ModuleNotFoundError: No module named 'sklearn'

# Vision

## Sub Heading 1

In [ ]:
# code here...

# Multi-modal

## Sub Heading 1

In [ ]:
# code here

# Final Output

In [ ]:
# code